# `gee_s1s2_translator` — harvest walkthrough

End-to-end demonstration of Phase 1 on a small subset (one AOI, one
date window). Mirrors what the CLI does, in cells you can step through
interactively while you set up credentials.

Prerequisites: see the README's 'Setup' section.

1. Earth Engine project registered
2. GCS bucket created
3. `.env` populated
4. `pip install -e ".[notebook]"` from the repo root

## 1. Setup and authentication

In [ ]:
from pathlib import Path
from gee_s1s2.auth import (
    load_env, init_earthengine, get_gcs_client, check_gcs_bucket,
)
from gee_s1s2.config import load_config

load_env()
init_earthengine()
client = get_gcs_client(); check_gcs_bucket(client)
config = load_config(Path('config/operational_v1.yaml'))
print(config.project.name)

## 2. Build one AOI geometry and inspect on a folium map

In [ ]:
from gee_s1s2.aois import aoi_geometry, shapely_to_ee
import folium

aoi = config.aoi_by_name('TBH SPA training area')
geom = aoi_geometry(aoi)
centroid = geom.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)
folium.GeoJson(geom.__geo_interface__).add_to(m)
m

## 3. Build the S1 GRD + S2 SR collections for one date window

In [ ]:
from gee_s1s2.catalogue import s1_grd_collection, s2_sr_collection
from gee_s1s2.calibration import calibrate_grd_collection

geom_ee = shapely_to_ee(geom)
window = config.window_by_label('comparison 2024')

s1_raw = s1_grd_collection(config.sentinel1, geom_ee, window.start, window.end)
s1_calibrated = calibrate_grd_collection(
    s1_raw, config.calibration, config.sentinel1.polarisations)
s2 = s2_sr_collection(config.sentinel2, geom_ee, window.start, window.end)
print('S1 candidates:', s1_raw.size().getInfo())
print('S2 candidates:', s2.size().getInfo())

## 4. Pair S1 and S2 acquisitions, run the AOI cloud check

In [ ]:
from gee_s1s2.pairing import find_pairs, materialise_s1, materialise_s2
from gee_s1s2.filtering import aoi_cloud_pct

s1_items = materialise_s1(s1_calibrated)
s2_items = materialise_s2(s2)
pairs = find_pairs('TBH SPA training area', s1_items, s2_items, config.pairing)
print(f'{len(pairs)} candidate pairs')
for p in pairs[:5]:
    print(f'  {p.s1.id[:40]}.. <-> {p.s2.id[:50]}.. sep={p.separation_days:.2f}d')

## 5. Submit one TFRecord export task to GCS

In [ ]:
from gee_s1s2.export import export_patches_for_bucket, stack_pair_image
from gee_s1s2.auth import get_gcs_bucket, get_gcs_prefix
import ee

if pairs:
    p = pairs[0]
    s1_img = s1_calibrated.filter(ee.Filter.eq('system:index', p.s1.id)).first()
    s2_img = s2.filter(ee.Filter.eq('system:index', p.s2.id)).first()
    stacked = stack_pair_image(s1_img, s2_img,
                                config.sentinel2.bands,
                                config.sentinel1.polarisations)
    task = export_patches_for_bucket(
        stacked, geom_ee, p.aoi_name, 'comparison 2024', [p.pair_id],
        config, get_gcs_bucket(), get_gcs_prefix(), split='train',
    )
    print(f'Task submitted; state={task.state}')
    print(f'Output: gs://{task.bucket}/{task.file_prefix}*.tfrecord.gz')

## 6. Open the resulting TFRecord and inspect one paired patch

Run this only after the task has reached COMPLETED.

In [ ]:
# Wait for task and fetch the local copy via gsutil/gcloud or google-cloud-storage.
# Then:
#   from gee_s1s2.export import open_tfrecord_local
#   ds = open_tfrecord_local('local_cache/sample.tfrecord.gz',
#                             ['VV','VH','B02','B03','B04','B08','B11','B12'])
#   for example in ds.take(1):
#       print({k: v.shape for k, v in example.items()})
print('See README section: "Verifying the TFRecord output".')